# Notebook 04: Test the Deployed Serverless Endpoint

**Module:** ITI113 Machine Learning & Operations  
**Focus Area:** C - MLOps & Deployment  

---

## What this notebook does

0. Configuration and discovery of team03's deployed SageMaker Serverless Endpoint
1. Confirms the endpoint is InService and ready to invoke
2. Inspects the deployed model artifact and traces it back to the SageMaker Model Registry package it was approved from
3. Tests the endpoint with boto3, using the exact request and response shape the Streamlit app uses
4. Tests batch invocation with several messages at once

This notebook does not build a UI. The project's demo app is Streamlit, pages/1_Scam_Detector.py, which runs as its own process via streamlit run app.py. This notebook validates the endpoint and settles on the exact invoke_endpoint call the Streamlit app uses; that wiring itself happens directly in pages/1_Scam_Detector.py, not here.

## 0. Configuration and Endpoint Discovery

Searches for serverless endpoints created by this team, matching TEAM_ID in the endpoint name, rather than hardcoding the endpoint name.

In [1]:
import boto3
import json

REGION = "ap-southeast-1"
TEAM_ID = "team03"
STUDENT_ID = "s301"
PROJECT_NAME = "crypto-scam-detector"

# Matches the naming used in Notebook 03.
ENDPOINT_NAME = f"iti113-{TEAM_ID}-{PROJECT_NAME}"

sts = boto3.client("sts", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

print(f"Region: {REGION}")
print(f"Team ID: {TEAM_ID}")
print(f"Student ID: {STUDENT_ID}")
print(f"Expected endpoint name: {ENDPOINT_NAME}")
print(f"AWS identity: {sts.get_caller_identity()['Arn']}")

response = sm.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=100
)

team_endpoints = [
    ep for ep in response["Endpoints"]
    if TEAM_ID in ep["EndpointName"].lower()
]

print(f"\nEndpoints found for {TEAM_ID}: {len(team_endpoints)}")
for ep in team_endpoints:
    print(f"  {ep['EndpointName']} (status: {ep['EndpointStatus']})")

Region: ap-southeast-1
Team ID: team03
Student ID: s301
Expected endpoint name: iti113-team03-crypto-scam-detector
AWS identity: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team03/SageMaker

Endpoints found for team03: 1
  iti113-team03-crypto-scam-detector (status: InService)


## 1. Confirm the Serverless Endpoint Is Active

The endpoint must show InService before it can be invoked.

In [2]:
try:
    for ep in team_endpoints:
        ENDPOINT_NAME = ep["EndpointName"]
        endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        print(
            f"\nEndpoint name: {endpoint_desc['EndpointName']}\n"
            f"Status: {endpoint_desc['EndpointStatus']}\n"
            f"Creation time: {endpoint_desc['CreationTime']}\n"
            f"Last modified: {endpoint_desc['LastModifiedTime']}"
        )

    if not team_endpoints:
        print(
            f"No endpoints found for {TEAM_ID}. Run Notebook 03's deploy section first, "
            "then rerun the discovery cell above."
        )
except Exception as e:
    print("Unable to describe endpoint.")
    print("Check that the endpoint exists and that your role has permission to access it.")
    print(f"Error: {e}")


Endpoint name: iti113-team03-crypto-scam-detector
Status: InService
Creation time: 2026-08-15 15:52:47.554000+00:00
Last modified: 2026-08-15 15:55:20.714000+00:00


## 2. Inspect the Endpoint's Model Artifact (Optional)

Traces the deployed endpoint back to the SageMaker Model Registry package it was approved from, useful for the Final Report's traceability discussion linking dataset, preprocessing, training job, model registry and endpoint.

In [3]:
for ep in team_endpoints:
    ENDPOINT_NAME = ep["EndpointName"]
    print(f"Endpoint name: {ENDPOINT_NAME}")

    try:
        endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        endpoint_config_name = endpoint_desc["EndpointConfigName"]
        print(f"Endpoint status: {endpoint_desc['EndpointStatus']}\nEndpoint config: {endpoint_config_name}")

        endpoint_config = sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
        production_variants = endpoint_config.get("ProductionVariants", [])

        if not production_variants:
            print("No production variants found.")
            continue

        for variant in production_variants:
            variant_name = variant.get("VariantName")
            model_name = variant.get("ModelName")
            print(f"Variant name: {variant_name}\nModel name: {model_name}")

            model_desc = sm.describe_model(ModelName=model_name)
            containers = model_desc.get("Containers") or model_desc.get("PrimaryContainer")

            if isinstance(containers, dict):
                containers = [containers]

            for container in containers or []:
                model_package_arn = container.get("ModelPackageName")
                if model_package_arn:
                    package_desc = sm.describe_model_package(ModelPackageName=model_package_arn)
                    print(f"Model package: {model_package_arn}\nModel package status: {package_desc.get('ModelApprovalStatus')}")
                else:
                    print(f"Model artifact (ModelDataUrl): {container.get('ModelDataUrl')}")

    except Exception as e:
        print("Could not inspect this endpoint's model chain.")
        print(f"Error: {e}")

Endpoint name: iti113-team03-crypto-scam-detector
Endpoint status: InService
Endpoint config: iti113-team03-crypto-scam-detector
Variant name: AllTraffic
Model name: team03-CryptoScamDetector-2026-08-15-15-52-46-126


Model package: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team03-CryptoScamDetector/37
Model package status: Approved


## 3. Test Endpoint Invocation with boto3

The endpoint accepts JSON input with the raw message text, the same shape pages/1_Scam_Detector.py will send: a text field holding the message string. It returns a list with one object containing prediction, label and probability. inference.py from Notebook 03 performs the cleaning, engineered-feature extraction and TF-IDF transform internally, so callers only ever send raw text.

In [4]:
def invoke_scam_detector(text, endpoint_name=ENDPOINT_NAME):
    """Invoke the deployed crypto-scam-detector endpoint with a single raw message.

    This is the exact call pages/1_Scam_Detector.py makes once wired up, validated here first and then reused in the Streamlit app.
    """
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps({"text": text}),
    )
    return json.loads(response["Body"].read())[0]


scam_message = (
    "URGENT: Your wallet has been selected for a guaranteed 100% profit airdrop! "
    "Deposit 500 USDT to your wallet address within 1 hour to claim now. "
    "Contact us on Telegram immediately, don't miss out!"
)

result = invoke_scam_detector(scam_message)
print(f"SCAM-STYLE MESSAGE\nPrediction: {result['label']}\nProbability: {result['probability']:.1%}")

SCAM-STYLE MESSAGE
Prediction: Scam
Probability: 99.9%


In [5]:
legit_message = (
    "Been dollar-cost averaging into ETH for about a year now, curious what "
    "everyone's thoughts are on the current market conditions."
)

borderline_message = (
    "Hey, our community wallet is doing a small giveaway this week, check the "
    "pinned post in the group for details."
)

for label, msg in [("LEGIT-STYLE MESSAGE", legit_message), ("BORDERLINE / AMBIGUOUS MESSAGE", borderline_message)]:
    result = invoke_scam_detector(msg)
    print(f"{label}\nPrediction: {result['label']}\nProbability: {result['probability']:.1%}\n")

LEGIT-STYLE MESSAGE
Prediction: Legitimate
Probability: 10.5%



BORDERLINE / AMBIGUOUS MESSAGE
Prediction: Legitimate
Probability: 20.9%



## 4. Batch Invocation

inference.py also accepts an instances list holding multiple text objects in a single request, useful for testing several examples at once or scoring a batch.

In [6]:
batch_messages = [
    scam_message,
    legit_message,
    borderline_message,
    "Congratulations! You have been selected to receive a free NFT, claim your prize now before it expires!",
    "Anyone else having trouble syncing their hardware wallet after the latest firmware update?",
]

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps({"instances": [{"text": t} for t in batch_messages]}),
)

results = json.loads(response["Body"].read())

print(f"{'Message (truncated)':<70} {'Label':<12} Probability")
for msg, res in zip(batch_messages, results):
    truncated = (msg[:65] + "...") if len(msg) > 65 else msg
    print(f"{truncated:<70} {res['label']:<12} {res['probability']:.1%}")

Message (truncated)                                                    Label        Probability
URGENT: Your wallet has been selected for a guaranteed 100% profi...   Scam         99.9%
Been dollar-cost averaging into ETH for about a year now, curious...   Legitimate   10.5%
Hey, our community wallet is doing a small giveaway this week, ch...   Legitimate   20.9%
Congratulations! You have been selected to receive a free NFT, cl...   Scam         85.8%
Anyone else having trouble syncing their hardware wallet after th...   Legitimate   5.0%
